In [1]:
# ================================================================
#  LCCN-R16a - TRANSDUCTIVE EXPOSURE OF THE LEXICAL RESOURCES  [CPU, ~1 h]
#  Answers IJIES Reviewer 1, point 6.
#  The KEEL tables (T1 noisy forms, T2 synonyms, T3 synsets, form->emotion)
#  were built from the one-million-comment collection from which the
#  40,000 annotated comments were later drawn; label evidence, bucket
#  elections and frequency counts are already training-split only.
#  Part A quantifies the exposure: lexicon entries whose form occurs in
#    validation or test comments but NOT in training comments.
#  Part B builds the train-only variant: those entries are removed from
#    T1, T2, T3, the emotion map and the negated-phrase map, so nothing
#    learnable only from evaluation comments remains; LCCN is rebuilt on
#    it (lccn_trainonly), with unified accounting and LR / SVM. Writes
#    trainonly_texts.csv for the GPU notebook LCCN-R16b (XLM-R, 3 seeds).
#  Add Input: paper1-nb1-lccn-final (output), paper1-nb2-lccn-ml (output,
#             trainer.py), final-emotion-dataset, gold-ds.
#  Accelerator None. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq indic-nlp-library')
print("python", sys.version.split()[0])


+ pip install -q regex emoji wordfreq indic-nlp-library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 5.0 MB/s eta 0:00:00
python 3.13.15


In [2]:
# ================================================================
#  CELL 1 - CERTIFIED PIPELINE (v5.3, selected explicitly)
# ================================================================
import shutil, importlib, sys, glob, re, os
for p in ("/kaggle/working/pipeline.py",):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("/kaggle/working/__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)
cand = sorted(glob.glob("/kaggle/input/**/pipeline.py", recursive=True))
assert cand, "attach the paper1-nb1-lccn-final output"
chosen = None
for p in cand:
    m = re.search(r'PIPELINE_VERSION\s*=\s*"([^"]+)"',
                  open(p, encoding="utf-8", errors="replace").read())
    v = m.group(1) if m else "?"
    print(f"  v{v:<5} {p}")
    if v == "5.3" and chosen is None: chosen = p
assert chosen, "no v5.3 pipeline.py among the inputs"
shutil.copy(chosen, "/kaggle/working/pipeline.py")
if "/kaggle/working" not in sys.path: sys.path.insert(0, "/kaggle/working")
import pipeline
FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3" and FP == "6a536dd437ba85b0"


  v5.3   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True


In [3]:
# ================================================================
#  CELL 2 - SHARED SETUP (replicates NB1 CELL 2; frozen split asserted)
# ================================================================
import importlib
importlib.reload(pipeline)
from pipeline import *
import pandas as pd, numpy as np
from collections import Counter

df, ID, TXT, LAB, RATERS = load_corpus()
norm_map, F2E, LEX = build_keel()
try:
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    _n = IndicNormalizerFactory().get_normalizer("kn")
    _n.normalize("\u0c95"); pipeline._indic_norm = _n; print("IndicNLP: FULL")
except Exception:
    import unicodedata
    class _F:
        def normalize(self, s): return unicodedata.normalize("NFC", s)
    pipeline._indic_norm = _F(); print("IndicNLP: FALLBACK(NFC)")

df["text_raw"] = df[TXT].map(lambda t: " ".join(process_comment(t)[0]))
SPLIT, TEST_HASH = make_split(df, ID, LAB)
assert TEST_HASH == "e2d6e1f7d564cace", f"split drift: {TEST_HASH}"
train = df[df[ID].isin(SPLIT["train"])]
evidence = LabelEvidence(train, TXT, LAB)
TRAIN_FREQ = Counter()
for t in train.text_raw: TRAIN_FREQ.update(w.lower() for w in t.split())
pipeline.TRAIN_FREQ = TRAIN_FREQ
print(f"train vocab {len(TRAIN_FREQ):,} | hash {TEST_HASH}")


corpus: 40,000 -> 40,000 after audit (dedup + conflict removal)
majority
disgust    10000
happy      10000
sad        10000
anger      10000
  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
IndicNLP: FULL
train  28,000
val     6,000
test    6,000
leakage check passed | test-set id hash: e2d6e1f7d564cace
train vocab 42,985 | hash e2d6e1f7d564cace


In [4]:
# ================================================================
#  CELL 3 - PART A: EXPOSURE STATISTICS
# ================================================================
from collections import Counter, defaultdict
def types_of(frame):
    v = set()
    for t in frame["text_raw"]:
        for w in t.split(): v.add(w.lower())
    return v
tr_mask = df[ID].isin(SPLIT["train"]); va_mask = df[ID].isin(SPLIT["val"]); te_mask = df[ID].isin(SPLIT["test"])
V_tr, V_va, V_te = types_of(df[tr_mask]), types_of(df[va_mask]), types_of(df[te_mask])
V_eval = V_va | V_te
SUSPECT = V_eval - V_tr                      # forms seen only in evaluation comments
print(f"types: train {len(V_tr):,} | val {len(V_va):,} | test {len(V_te):,} | val+test only (not in train) {len(SUSPECT):,}")

T1, T2, T3, EMO = set(LEX.T1), set(LEX.T2), set(LEX.T3), set(LEX.emo)
LEXK = T1 | T2 | T3
def share(a, b): return 100 * len(a & b) / max(len(a), 1)
print("\nshare of evaluation vocabulary types covered by the lexicon:")
for name, V in (("val", V_va), ("test", V_te)):
    print(f"  {name:<5} in T1 {share(V, T1):5.2f}%  in T1|T2|T3 {share(V, LEXK):5.2f}%  with emotion {share(V, EMO):5.2f}%")
print(f"  train in T1 {share(V_tr, T1):5.2f}%  in T1|T2|T3 {share(V_tr, LEXK):5.2f}%  with emotion {share(V_tr, EMO):5.2f}%")

print("\nlexicon entries attested in this corpus:")
for name, K in (("T1 noisy forms", T1), ("T2 synonyms", T2), ("T3 synsets", T3), ("form->emotion", EMO)):
    if not K:
        print(f"  {name:<16}        0 entries (table empty in this configuration)"); continue
    in_tr = len(K & V_tr); in_eval_only = len(K & SUSPECT); in_any = len(K & (V_tr | V_eval)); n = len(K)
    print(f"  {name:<16} {n:>8,} entries | attested in train {in_tr:>6,} ({100*in_tr/n:5.2f}%) | "
          f"attested ONLY in val/test {in_eval_only:>5,} ({100*in_eval_only/n:5.2f}%) | not attested in the 40K {n-in_any:>8,} ({100*(n-in_any)/n:5.2f}%)")

# how often do the evaluation-only entries actually fire on evaluation text?
fire = Counter()
for t in df.loc[va_mask | te_mask, "text_raw"]:
    for w in t.split():
        if w.lower() in SUSPECT and w.lower() in LEXK: fire["lexicon"] += 1
        if w.lower() in SUSPECT and w.lower() in EMO: fire["emotion"] += 1
tot_eval_tokens = sum(len(t.split()) for t in df.loc[va_mask | te_mask, "text_raw"])
print(f"\nevaluation-only forms that the lexicon can resolve: {len(SUSPECT & LEXK):,} types; "
      f"they occur {fire['lexicon']:,} times in val+test ({100*fire['lexicon']/tot_eval_tokens:.3f}% of evaluation tokens)")
print(f"evaluation-only forms with an emotion entry:          {len(SUSPECT & EMO):,} types; "
      f"{fire['emotion']:,} occurrences ({100*fire['emotion']/tot_eval_tokens:.3f}%)")
pd.DataFrame(sorted(SUSPECT & LEXK), columns=["form"]).to_csv("/kaggle/working/eval_only_lexicon_forms.csv", index=False)


types: train 42,985 | val 15,164 | test 15,588 | val+test only (not in train) 11,293

share of evaluation vocabulary types covered by the lexicon:
  val   in T1  8.26%  in T1|T2|T3  9.17%  with emotion 26.48%
  test  in T1  8.35%  in T1|T2|T3  9.25%  with emotion 26.37%
  train in T1  5.88%  in T1|T2|T3  6.29%  with emotion 19.61%

lexicon entries attested in this corpus:
  T1 noisy forms    126,567 entries | attested in train  2,529 ( 2.00%) | attested ONLY in val/test   376 ( 0.30%) | not attested in the 40K  123,662 (97.70%)
  T2 synonyms             0 entries (table empty in this configuration)
  T3 synsets            603 entries | attested in train    174 (28.86%) | attested ONLY in val/test    10 ( 1.66%) | not attested in the 40K      419 (69.49%)
  form->emotion     174,425 entries | attested in train  8,431 ( 4.83%) | attested ONLY in val/test 1,460 ( 0.84%) | not attested in the 40K  164,534 (94.33%)

evaluation-only forms that the lexicon can resolve: 386 types; they occur 4

In [5]:
# ================================================================
#  CELL 4 - PART B: TRAIN-ONLY RESOURCE VARIANT AND LCCN REBUILD
# ================================================================
import copy
from tqdm.auto import tqdm
LEX_T = copy.copy(LEX)
LEX_T.T1 = {k: v for k, v in LEX.T1.items() if k not in SUSPECT}
LEX_T.T2 = {k: v for k, v in LEX.T2.items() if k not in SUSPECT}
LEX_T.T3 = {k: v for k, v in LEX.T3.items() if k not in SUSPECT}
LEX_T.emo = {k: v for k, v in LEX.emo.items() if k not in SUSPECT}
LEX_T.phrase_emo = {k: v for k, v in LEX.phrase_emo.items() if k.split()[-1] not in SUSPECT}
LEX_T.stats = Counter()
norm_map_T = {k: v for k, v in norm_map.items() if k not in SUSPECT}
F2E_T = LEX_T.emo
print(f"removed from T1 {len(LEX.T1)-len(LEX_T.T1):,} | T2 {len(LEX.T2)-len(LEX_T.T2):,} | T3 {len(LEX.T3)-len(LEX_T.T3):,} | "
      f"emotion {len(LEX.emo)-len(LEX_T.emo):,} | phrases {len(LEX.phrase_emo)-len(LEX_T.phrase_emo):,} | norm_map {len(norm_map)-len(norm_map_T):,}")

single = {f: next(iter(e)) for f, e in F2E.items() if len(e) == 1}
df_tr, df_rest = df[tr_mask.values], df[~tr_mask.values]
def accounting(name, o):
    tr_out = [o.normalize(t) for t in tqdm(df_tr[TXT], desc=name, leave=False)]
    forms_rewritten = len({k[1] for k, v in o.cache.items() if v != k[1]})
    blocked = {k: v for k, v in o.stats.items() if k.startswith("blocked")}
    rest_out = [o.normalize(t) for t in df_rest[TXT]]
    ch = tot = 0
    for a, b in zip(list(df_tr["text_raw"]) + list(df_rest["text_raw"]), tr_out + rest_out):
        ra, rb = a.lower().split(), b.lower().split(); assert len(ra) == len(rb)
        tot += len(ra); ch += sum(1 for u, v in zip(ra, rb) if u != v)
    mp = defaultdict(Counter)
    for a, b in zip(df_tr["text_raw"], tr_out):
        for w, x in zip(a.split(), b.split()): mp[w.lower()][x.lower()] += 1
    buckets = defaultdict(set)
    for f, c in mp.items(): buckets[c.most_common(1)[0][0]].add(f)
    pairs = lab = same = 0
    for forms in buckets.values():
        n = len(forms)
        if n < 2: continue
        pairs += n*(n-1)//2
        ce = Counter(single[f] for f in forms if f in single); L = sum(ce.values())
        lab += L*(L-1)//2; same += sum(k*(k-1)//2 for k in ce.values())
    x = lab - same; prec = 1 - x/lab if lab else float("nan")
    full = pd.Series(tr_out + rest_out, index=list(df_tr.index) + list(df_rest.index)).reindex(df.index)
    # evaluation-side token change (val+test only), the quantity exposure could inflate
    ev_ch = ev_tot = 0
    for a, b in zip(df_rest["text_raw"], rest_out):
        ra, rb = a.lower().split(), b.lower().split(); ev_tot += len(ra); ev_ch += sum(1 for u, v in zip(ra, rb) if u != v)
    row = dict(arm=name, tokens_changed_full_pct=round(100*ch/tot, 2), tokens_changed_valtest_pct=round(100*ev_ch/ev_tot, 2),
               forms_rewritten_train=forms_rewritten, merged_pairs_train=pairs, x_cross_train=x,
               merge_precision_train=round(prec, 4), **blocked)
    print(f"{name:<16} tok {row['tokens_changed_full_pct']:5.2f}% (val+test {row['tokens_changed_valtest_pct']:5.2f}%)  "
          f"forms_rew {forms_rewritten:>6,}  pairs {pairs:>6,}  X {x}  prec {prec:.4f}")
    return row, full

rows = []
for name, (nm, ev_, lx, f2e) in (("lccn_full", (norm_map, evidence, LEX, F2E)),
                                 ("lccn_trainonly", (norm_map_T, evidence, LEX_T, F2E_T))):
    o = LCCN(nm, ev_, make_veto(f2e, lx), translit=None, lex=lx)
    row, texts = accounting(name, o); rows.append(row); df[f"text_{name}"] = texts.values
acc = pd.DataFrame(rows); acc.to_csv("/kaggle/working/trainonly_accounting.csv", index=False)
print("\n", acc.to_string(index=False))
diff = sum(1 for a, b in zip(df["text_lccn_full"], df["text_lccn_trainonly"]) if a != b)
print(f"\ncomments whose normalized text differs between full and train-only resources: {diff:,} of 40,000")
df[[ID, "text_lccn_full", "text_lccn_trainonly"]].to_csv("/kaggle/working/trainonly_texts.csv", index=False, encoding="utf-8-sig")
print("CONSISTENCY: lccn_full must print 14.16%, 4,680, 2,050, X 1.")


removed from T1 376 | T2 0 | T3 10 | emotion 1,460 | phrases 15 | norm_map 376


lccn_full:   0%|          | 0/28000 [00:00<?, ?it/s]

lccn_full        tok 14.16% (val+test 14.06%)  forms_rew  4,680  pairs  2,050  X 1  prec 0.9988


lccn_trainonly:   0%|          | 0/28000 [00:00<?, ?it/s]

lccn_trainonly   tok 14.06% (val+test 13.77%)  forms_rew  4,678  pairs  2,050  X 1  prec 0.9988

            arm  tokens_changed_full_pct  tokens_changed_valtest_pct  forms_rewritten_train  merged_pairs_train  x_cross_train  merge_precision_train  blocked_orthographic  blocked_gtest  blocked_veto_negation  blocked_veto_emotion  blocked_cluster  blocked_veto_length
     lccn_full                    14.16                       14.06                   4680                2050              1                 0.9988                  3321            252                     35                   102               32                    3
lccn_trainonly                    14.06                       13.77                   4678                2050              1                 0.9988                  3305            252                     35                    94               32                    3

comments whose normalized text differs between full and train-only resources: 459 of 40,000
CO

In [6]:
# ================================================================
#  CELL 5 - CLASSICAL MACRO-F1 (trainer v1.4)
# ================================================================
import shutil, glob, os, importlib
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)) if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)); assert tc
shutil.copy(tc[0], "/kaggle/working/trainer.py"); sys.modules.pop("trainer", None)
import trainer; importlib.reload(trainer); from trainer import *
assert trainer.TRAINER_VERSION == "1.4"; trainer.fingerprint()
sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
tr, va, te, HAVE = load_splits(sd[0])
for c in ("text_lccn_full", "text_lccn_trainonly"):
    lut = dict(zip(df[ID].astype(str), df[c]))
    for f in (tr, va, te):
        f[c] = f["id"].astype(str).map(lut); assert f[c].notna().all(), c
res = run_grid(tr, va, te, ["raw", "lccn", "lccn_full", "lccn_trainonly"], ["Logistic Regression", "SVM"],
               tag="lccn_r16_trainonly_classical", resume=False)
print(res.to_string(index=False))
r_lr, r_svm = res.iloc[0], res.iloc[1]
print(f"\nclassical mean: lccn (NB1 texts) {(float(r_lr['lccn'])+float(r_svm['lccn']))/2:.4f} | "
      f"lccn_full {(float(r_lr['lccn_full'])+float(r_svm['lccn_full']))/2:.4f} | "
      f"lccn_trainonly {(float(r_lr['lccn_trainonly'])+float(r_svm['lccn_trainonly']))/2:.4f}")
print("DONE - trainonly_texts.csv, trainonly_accounting.csv, eval_only_lexicon_forms.csv written; attach this output to R16b.")


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']

=== Logistic Regression / raw ===
  -> macro-F1 0.8033

=== Logistic Regression / lccn ===
  -> macro-F1 0.8017

=== Logistic Regres